In [13]:
from pathlib import Path
import pandas as pd
import numpy as np

# Locate the project root
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "public"
TRAIN_CSV = DATA_DIR / "train.csv"
TRAIN_IMAGES = DATA_DIR / "train"
TEST_IMAGES = DATA_DIR / "test"

assert TRAIN_CSV.exists(), f"Missing CSV: {TRAIN_CSV}"
assert TRAIN_IMAGES.is_dir(), f"Missing training images: {TRAIN_IMAGES}"

df = pd.read_csv(TRAIN_CSV)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Unique image IDs:", df["image_id"].nunique())
print("Defect classes:", sorted(df["defect_type"].dropna().unique()))
print("Training images:", len(list(TRAIN_IMAGES.glob("*.jpg"))))
print("Test images:", len(list(TEST_IMAGES.glob("*.jpg"))))

display(df.head())


Dataset shape: (45572, 5)
Columns: ['image_id', 'defect_type', 'mask_rle', 'width', 'height']
Unique image IDs: 11393
Defect classes: ['defect_a', 'defect_b', 'defect_c', 'defect_d']
Training images: 11393
Test images: 3000


,image_id,defect_type,mask_rle,width,height
0,4S22I7SA34,defect_a,7255 6 7377 12 7505 12 7633 12 7762 11 7890 11...,800,128
1,4S22I7SA34,defect_b,NaN,800,128
2,4S22I7SA34,defect_c,NaN,800,128
3,4S22I7SA34,defect_d,NaN,800,128
4,DZ899111OE,defect_a,NaN,800,128


In [14]:
# Image dimensions
height, width = 8, 8

# Highlighted pixel indices from the question (1-based)
positive_indices = [12, 13, 23, 24, 25, 26, 44, 45, 47]

# Create a flattened binary mask
mask = np.zeros(height * width, dtype=np.uint8)

# Convert 1-based indices to 0-based indices
mask[np.array(positive_indices) - 1] = 1

# Reshape using column-major (Fortran) order
mask_2d = mask.reshape((height, width), order="F")

print("Binary mask:")
print(mask_2d)


# RLE encoding
def rle_encode(mask):
    pixels = mask.flatten(order="F")

    pixels = np.concatenate([[0], pixels, [0]])
    changes = np.where(pixels[1:] != pixels[:-1])[0]

    starts = changes[::2] + 1
    lengths = changes[1::2] - changes[::2]

    return " ".join(
        str(value)
        for pair in zip(starts, lengths)
        for value in pair
    )


rle_string = rle_encode(mask_2d)

print("\nRLE String:", rle_string)

Binary mask:
[[0 0 0 1 0 0 0 0]
 [0 0 0 1 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 1 0 0 0 1 0 0]
 [0 1 0 0 0 1 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 1 0 0 1 0 0]
 [0 0 1 0 0 0 0 0]]

RLE String: 12 2 23 4 44 2 47 1


In [15]:
IMAGE_ID = "0ZUU8LH57Y"
HEIGHT, WIDTH = 128, 800

# Step 1: Get all four defect rows for this image
image_rows = df[df["image_id"] == IMAGE_ID]

print("Defect rows:")
display(image_rows[["image_id", "defect_type", "mask_rle"]])


# Step 2: RLE decoding function
def rle_decode(rle, height, width):

    # Return an all-zero mask for missing or empty RLE
    mask = np.zeros(height * width, dtype=np.uint8)

    if pd.isna(rle) or str(rle).strip() == "":
        return mask.reshape((height, width), order="F")

    # Convert RLE string into integer values
    values = np.asarray(rle.split(), dtype=np.int64)

    # Convert 1-based positions to 0-based positions
    starts = values[0::2] - 1
    lengths = values[1::2]

    # Fill defect regions with 1
    for start, length in zip(starts, lengths):
        mask[start:start + length] = 1

    # Reshape using column-major order
    return mask.reshape((height, width), order="F")


# Step 3: Decode each defect mask and count positive pixels
total_positive_pixels = 0

for _, row in image_rows.iterrows():

    mask = rle_decode(
        row["mask_rle"],
        HEIGHT,
        WIDTH
    )

    pixel_count = int(mask.sum())

    print(f'{row["defect_type"]}: {pixel_count}')

    total_positive_pixels += pixel_count


# Step 4: Print final answer
print("\nTotal positive pixels:", total_positive_pixels)

Defect rows:


,image_id,defect_type,mask_rle
40804,0ZUU8LH57Y,defect_a,88169 11 88183 6 88294 24 88422 24 88550 24 88...
40805,0ZUU8LH57Y,defect_b,NaN
40806,0ZUU8LH57Y,defect_c,37642 1 37769 2 37897 3 38025 4 38153 5 38281 ...
40807,0ZUU8LH57Y,defect_d,79113 13 79239 16 79367 17 79495 18 79622 21 7...


defect_a: 250
defect_b: 0
defect_c: 1232
defect_d: 5032

Total positive pixels: 6514


In [17]:
# Step 1: Keep rows where mask_rle is not missing or empty
valid_rows = df[
    df["mask_rle"].notna() &
    (df["mask_rle"].astype(str).str.strip() != "")
]

# Step 2: Count unique images for each defect class
defect_counts = (
    valid_rows.groupby("defect_type")["image_id"]
    .nunique()
    .sort_values(ascending=False)
)

print("Defect occurrence counts:")
print(defect_counts)

# Step 3: Identify most common and rarest classes
most_common = defect_counts.idxmax()
most_common_count = defect_counts.max()

rarest = defect_counts.idxmin()
rarest_count = defect_counts.min()

print("\nMost common:", most_common, most_common_count)
print("Rarest:", rarest, rarest_count)

Defect occurrence counts:
defect_type
defect_c    4100
defect_d    1707
defect_a    1081
defect_b     313
Name: image_id, dtype: int64

Most common: defect_c 4100
Rarest: defect_b 313


In [18]:
# Step 1: Identify whether each defect is present
df["present"] = (
    df["mask_rle"].notna() &
    (df["mask_rle"].astype(str).str.strip() != "")
)

# Step 2: Create an image-level defect presence matrix
presence_matrix = df.pivot_table(
    index="image_id",
    columns="defect_type",
    values="present",
    aggfunc="any",
    fill_value=False
)

# Step 3: Calculate the 4x4 co-occurrence matrix
co_occurrence = (
    presence_matrix.astype(int).T @
    presence_matrix.astype(int)
)

print("Defect Co-occurrence Matrix:")
display(co_occurrence)

# Step 4: Find the pair with maximum co-occurrence
pairs = []

classes = co_occurrence.columns.tolist()

for i in range(len(classes)):
    for j in range(i + 1, len(classes)):
        pairs.append((
            classes[i],
            classes[j],
            co_occurrence.iloc[i, j]
        ))

most_common_pair = max(pairs, key=lambda x: x[2])

print("\nMost co-occurring pair:",
      most_common_pair[0], most_common_pair[1])

print("Co-occurrence count:", most_common_pair[2])

Defect Co-occurrence Matrix:


defect_type,defect_a,defect_b,defect_c,defect_d
defect_type,,,,
defect_a,1081,42,310,234
defect_b,42,313,95,64
defect_c,310,95,4100,705
defect_d,234,64,705,1707



Most co-occurring pair: defect_c defect_d
Co-occurrence count: 705


In [19]:
# Step 1: Calculate the Pearson correlation matrix
correlation_matrix = presence_matrix.astype(int).corr(method="pearson")

print("Pearson Correlation Matrix:")
display(correlation_matrix)

# Step 2: Extract correlation between defect_c and defect_d
correlation_value = correlation_matrix.loc["defect_c", "defect_d"]

# Step 3: Round to 2 decimal places
correlation_value = round(correlation_value, 2)

print("\nCorrelation between defect_c and defect_d:",
      correlation_value)

Pearson Correlation Matrix:


defect_type,defect_a,defect_b,defect_c,defect_d
defect_type,,,,
defect_a,1.000000,0.022541,-0.049311,0.060452
defect_b,0.022541,1.000000,-0.019735,0.025733
defect_c,-0.049311,-0.019735,1.000000,0.046475
defect_d,0.060452,0.025733,0.046475,1.000000



Correlation between defect_c and defect_d: 0.05


In [20]:
from itertools import combinations

HEIGHT, WIDTH = 128, 800

# Step 1: RLE decoding function
def rle_decode(rle, height, width):
    mask = np.zeros(height * width, dtype=np.uint8)

    if pd.isna(rle) or str(rle).strip() == "":
        return mask.reshape((height, width), order="F")

    values = np.asarray(rle.split(), dtype=np.int64)

    starts = values[0::2] - 1
    lengths = values[1::2]

    for start, length in zip(starts, lengths):
        mask[start:start + length] = 1

    return mask.reshape((height, width), order="F")


# Step 2: Identify images containing multiple defects
valid_rows = df[
    df["mask_rle"].notna() &
    (df["mask_rle"].astype(str).str.strip() != "")
]

multi_label_ids = (
    valid_rows.groupby("image_id")["defect_type"]
    .nunique()
)

multi_label_ids = multi_label_ids[multi_label_ids > 1].index

# Step 3: Calculate pairwise pixel intersections
max_overlap_pixels = 0
max_overlap_pair = None

for image_id in multi_label_ids:

    image_rows = valid_rows[valid_rows["image_id"] == image_id]

    masks = {}

    for _, row in image_rows.iterrows():
        masks[row["defect_type"]] = rle_decode(
            row["mask_rle"], HEIGHT, WIDTH
        )

    for class_a, class_b in combinations(masks.keys(), 2):

        overlap = int(np.logical_and(
            masks[class_a],
            masks[class_b]
        ).sum())

        if overlap > max_overlap_pixels:
            max_overlap_pixels = overlap
            max_overlap_pair = (class_a, class_b)

# Step 4: Print the result
print("Maximum overlapping pixels:", max_overlap_pixels)
print("Defect pair:", max_overlap_pair)
print("Spatially mutually exclusive:",
      "Yes" if max_overlap_pixels == 0 else "No")

Maximum overlapping pixels: 0
Defect pair: None
Spatially mutually exclusive: Yes


In [23]:
import cv2
import numpy as np
from torch.utils.data import Dataset, DataLoader

HEIGHT, WIDTH = 128, 800

DEFECT_CLASSES = [
    "defect_a",
    "defect_b",
    "defect_c",
    "defect_d"
]


# RLE decoding function
def rle_decode(rle, height, width):

    mask = np.zeros(height * width, dtype=np.uint8)

    if pd.isna(rle) or str(rle).strip() == "":
        return mask.reshape((height, width), order="F")

    values = np.asarray(rle.split(), dtype=np.int64)

    starts = values[0::2] - 1
    lengths = values[1::2]

    for start, length in zip(starts, lengths):
        mask[start:start + length] = 1

    return mask.reshape((height, width), order="F")


# Custom PyTorch Dataset
class SurfaceAnomalyDataset(Dataset):

    def __init__(self, dataframe, image_dir):
        self.df = dataframe
        self.image_dir = image_dir

        self.image_ids = dataframe["image_id"].unique()

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):

        image_id = self.image_ids[idx]

        # Load image
        image_path = self.image_dir / f"{image_id}.jpg"

        image = cv2.imread(str(image_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        image = cv2.resize(image, (WIDTH, HEIGHT))
        image = image.astype(np.float32) / 255.0

        # Convert image to CHW format
        image = torch.tensor(
            image.transpose(2, 0, 1),
            dtype=torch.float32
        )

        # Get corresponding defect rows
        rows = self.df[self.df["image_id"] == image_id]

        masks = []

        for defect in DEFECT_CLASSES:

            defect_row = rows[rows["defect_type"] == defect]

            if len(defect_row) == 0:
                rle = None
            else:
                rle = defect_row.iloc[0]["mask_rle"]

            mask = rle_decode(rle, HEIGHT, WIDTH)
            masks.append(mask)

        # Stack four masks into four channels
        masks = np.stack(masks, axis=0)

        masks = torch.tensor(
            masks,
            dtype=torch.float32
        )

        return image, masks

In [24]:
dataset = SurfaceAnomalyDataset(df, TRAIN_IMAGES)

dataloader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=False
)

# Extract first batch
images, masks = next(iter(dataloader))

print("Images shape:", images.shape)
print("Masks shape:", masks.shape)

print("Images dtype:", images.dtype)
print("Masks dtype:", masks.dtype)

Images shape: torch.Size([4, 3, 128, 800])
Masks shape: torch.Size([4, 4, 128, 800])
Images dtype: torch.float32
Masks dtype: torch.float32


In [25]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim

# Split by unique image IDs to avoid data leakage
image_ids = df["image_id"].unique()

train_ids, val_ids = train_test_split(
    image_ids,
    test_size=0.2,
    random_state=42
)

train_df = df[df["image_id"].isin(train_ids)].copy()
val_df = df[df["image_id"].isin(val_ids)].copy()

train_dataset = SurfaceAnomalyDataset(train_df, TRAIN_IMAGES)
val_dataset = SurfaceAnomalyDataset(val_df, TRAIN_IMAGES)

train_loader = DataLoader(
    train_dataset, batch_size=4, shuffle=True
)

val_loader = DataLoader(
    val_dataset, batch_size=4, shuffle=False
)

In [26]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc1 = ConvBlock(3, 32)
        self.enc2 = ConvBlock(32, 64)
        self.enc3 = ConvBlock(64, 128)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = ConvBlock(128, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec3 = ConvBlock(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec2 = ConvBlock(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.dec1 = ConvBlock(64, 32)

        self.output = nn.Conv2d(32, 4, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        b = self.bottleneck(self.pool(e3))

        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.output(d1)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet().to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=3e-4)

train_losses = []
val_losses = []

best_val_loss = float("inf")
best_epoch = 0

for epoch in range(5):

    # Training
    model.train()
    running_train_loss = 0.0

    for images, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, masks)

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()

    epoch_train_loss = running_train_loss / len(train_loader)

    # Validation
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)

            running_val_loss += loss.item()

    epoch_val_loss = running_val_loss / len(val_loader)

    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)

    print(
        f"Epoch {epoch + 1}/5 | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Validation Loss: {epoch_val_loss:.4f}"
    )

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_epoch = epoch + 1
        torch.save(model.state_dict(), "best_model.pth")

print("\nBest Epoch:", best_epoch)
print("Lowest Validation Loss:", round(best_val_loss, 4))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.plot(range(1, 6), train_losses, marker="o", label="Training Loss")
plt.plot(range(1, 6), val_losses, marker="o", label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.xticks(range(1, 6))
plt.legend()
plt.grid(True)
plt.show()